# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [2]:
import pathlib
import numpy
import pandas
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [3]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:53716/status,
Dashboard: http://127.0.0.1:53716/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53717,Workers: 0
Dashboard: http://127.0.0.1:53716/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:54158,Total threads: 2
Dashboard: http://127.0.0.1:54162/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:53720,


# Values to edit

In [4]:
max_cloud_covers = [5, 1, 10, 0] # percentage
lowtide_search_range = 60 # days - +/- this devided by 2

In [5]:
site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [16]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
survey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"
mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"

# Normal 20% MNDWI threshold

In [9]:
mndwi_thresholds = pandas.read_csv(mndwi_file)
all_baselines = sentinel2.load_band_quality_baseline()

for max_cloud_cover in max_cloud_covers:
    print(f"\n\nMax cloud percentage {max_cloud_cover}, MNDWI filtering to select lowish tide")
    for site in site_names:

        mndwi_threshold = mndwi_thresholds.loc[mndwi_thresholds["site"] == site, "threshold",].squeeze()
        site_baseline = all_baselines.loc[all_baselines["site"] == site]
        if not isinstance(mndwi_threshold, float):
            print(f"WARNING: Expected a float tide threshold for {site}. Exiting.")
            break
        if len(site_baseline) != len(sentinel2.RGB_BANDS):
            print(f"WARNING: Expected a float tide threshold for {site}. Exiting.")
            break
        for i in range(2):
            try:
                sampling.get_site_satellite_mndwi(
                    site_name=site,
                    survey_dates_file=servey_dates_file,
                    lowtide_search_range=lowtide_search_range,
                    uav_folder=uav_folder,
                    max_cloud_cover=max_cloud_cover,
                    mndwi_threshold=mndwi_threshold,
                    anomaly_filter_baseline=site_baseline,
                    n_mad=3,
                )
                break
            except Exception:
                print("Caught exception downloading images and try again.")
        print(f"Save RGB for {site}")
        satellite_images_path = utils.get_training_satellite_path(
            site_name=site,
            use_threshold=True, 
            max_cloud_cover=max_cloud_cover,
            filtered=True)
        utils.rgb_from_satellite(satellite_data_path=satellite_images_path)



Max cloud percentage 5, MNDWI filtering to select lowish tide
Site Ihutai
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2025-03-23/2025-05-22
	MNDWI thresholded tiles: 2 from 13 STAC items
	Low Cloud Tiles: 1 from the low tide tiles of 2. Cloud percentages [21.8  0. ] Kept [False  True]
		2025-04-12: B04=0.63, B03=0.76, B02=0.92 -> kept
	Anomaly filter: 1 of 1 dates kept for site Ihutai (n_mad=3)
Save RGB for Ihutai
Site CatlinsLake
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2022-07-11/2022-09-09
	MNDWI thresholded tiles: 5 from 34 STAC items
	Low Cloud Tiles: 0 from the low tide tiles of 5. Cloud percentages [63.  73.5 88.   7.9 65.6] Kept [False False False False False]
Save RGB for CatlinsLake
Satellite data path C:\Local\repos\seagrass-detection\data\training\tide_thresholded_cloud_5_filtered\satellite_images\CatlinsLake_sentinel-2.nc does not exist.
Site CatlinsRiverMouth
	Save satellite image of th

In [9]:
mndwi_thresholds

,site,threshold
0,Ihutai,0.285990
1,CatlinsLake,0.324971
2,CatlinsRiverMouth,0.077602
3,Childrens,0.351367
4,Duvauchelle,0.378407
5,Robinsons,0.321775
6,Takamatua,0.455245
